# INGENIERÍA DE VARIABLES

El notebook tiene como propósito crear nuevas variables útiles para el entrenamiento de un modelo predictivo , combina variables relacionadas con la ocupación, la superficie, el equipamiento, la iluminación, el aire acondicionado, el consumo histórico, la generación solar, los cortes eléctricos y las características del inmueble.

También genera indicadores de calidad, inconsistencias y divisiones no calculables, valida que las nuevas variables no contengan valores infinitos o nulos y finalmente las incorpora al DataFrame original. Como resultado, se obtiene un conjunto de datos enriquecido con características que pueden ayudar al modelo a identificar relaciones más complejas y mejorar su capacidad predictiva.


In [ ]:
import pandas as pd
import numpy as np

bd = pd.read_csv("/content/energia_v1_imputada_ohe(3).csv")

## 1. CREACIÓN DE VARIABLES

### Función de división segura

La función `division_segura()` permite calcular cocientes entre dos variables evitando errores o resultados inválidos cuando el denominador es cero, alguno de los valores es nulo o la operación produce valores infinitos.

Primero, el numerador y el denominador se convierten en objetos `pandas.Series` de tipo `float` y se alinean con el índice del DataFrame `bd`. Después, se crea una serie de resultados inicializada con el valor definido en `valor_por_defecto`, que por defecto es `0`.

La división se realiza únicamente en los registros que cumplen las siguientes condiciones:

```text
denominador no es nulo
y denominador es diferente de 0
y numerador no es nulo
```

En las filas que cumplen estas condiciones se calcula:

```text
resultado = numerador / denominador
```

En los registros donde la división no puede calcularse, se conserva el valor por defecto. Finalmente, cualquier resultado infinito o nulo que pudiera permanecer se reemplaza también por dicho valor.

La función retorna una serie numérica con el mismo índice de `bd`, lo que permite utilizarla directamente en la creación de nuevas variables derivadas sin generar divisiones entre cero, valores infinitos o valores faltantes.


In [ ]:
def division_segura(
    numerador,
    denominador,
    valor_por_defecto=0
):

    numerador = pd.Series(
        numerador,
        index=bd.index,
        dtype=float
    )

    denominador = pd.Series(
        denominador,
        index=bd.index,
        dtype=float
    )

    resultado = pd.Series(
        valor_por_defecto,
        index=bd.index,
        dtype=float
    )

    mascara_valida = (
        denominador.notna()
        & denominador.ne(0)
        & numerador.notna()
    )

    resultado.loc[mascara_valida] = (
        numerador.loc[mascara_valida]
        / denominador.loc[mascara_valida]
    )

    resultado = (
        resultado
        .replace(
            [np.inf, -np.inf],
            valor_por_defecto
        )
        .fillna(valor_por_defecto)
    )

    return resultado

### Inspección y resumen de las variables iniciales

Este bloque tiene como propósito revisar la disponibilidad, el tipo de dato, la cantidad de valores únicos y la presencia de valores faltantes en las variables que conforman el conjunto de datos antes de realizar el proceso de ingeniería de características.

En primer lugar, se establece una lista de variables iniciales que incluye:

* Variables de identificación.
* Características de la vivienda.
* Información sobre los residentes.
* Equipamiento y electrodomésticos.
* Patrones de uso de los espacios.
* Variables de iluminación y climatización.
* Información sobre cortes eléctricos y fuentes de respaldo.
* Variables de consumo energético.
* Variables derivadas del consumo.
* Banderas de inconsistencias y correcciones.
* Variables categóricas transformadas mediante One-Hot Encoding.
* La variable objetivo `perfil_energetico`.

## Funcionalidades que cumple

### 1. Definición de las variables que deben inspeccionarse

El bloque establece de manera explícita cuáles columnas se espera encontrar en el DataFrame.

Esta lista funciona como referencia para comprobar que la base contenga tanto las variables originales como las variables generadas durante las fases anteriores de limpieza, imputación y codificación.

### 2. Configuración del número de valores que se mostrarán

Se establece un límite general de 15 valores únicos por variable para evitar que la salida sea excesivamente extensa.

Para las variables `perfil_energetico` y `certificacion_energetica_previa` se permite mostrar hasta 30 valores, debido a que se considera conveniente inspeccionar una mayor cantidad de categorías o valores posibles.

### 3. Comprobación de columnas disponibles

El bloque compara la lista de variables esperadas con las columnas existentes en `bd`.

Como resultado, separa las variables en dos grupos:

* Variables que están disponibles y pueden analizarse.
* Variables que no se encuentran en el DataFrame.

Esta comprobación evita errores al intentar consultar columnas inexistentes.

### 4. Inspección individual de cada variable disponible

Para cada variable encontrada se obtiene la siguiente información:

* Posición de la variable dentro del recorrido.
* Nombre de la variable.
* Tipo de dato.
* Cantidad de valores únicos sin contar los valores nulos.
* Cantidad de valores únicos incluyendo los valores nulos.
* Número total de valores faltantes.
* Límite de valores únicos que se mostrarán.
* Muestra de los primeros valores únicos encontrados.

### 5. Eliminación de valores repetidos para la inspección

Antes de presentar los valores de una variable, se conserva una sola aparición de cada valor.

Esta operación permite observar las categorías o valores diferentes presentes en la columna sin repetirlos.

### 6. Ordenamiento de los valores únicos

El bloque intenta ordenar los valores únicos y colocar los valores nulos al final.

Cuando una variable contiene tipos de datos incompatibles entre sí y no puede ordenarse, el proceso continúa utilizando el orden disponible, evitando que la inspección se detenga por un error.

### 7. Control de salidas extensas

Cuando una variable contiene más valores únicos que el límite establecido, solo se muestran los primeros valores permitidos.

También se informa que la salida fue limitada y se indica:

* Cuántos valores se muestran.
* Cuántos valores únicos existen realmente.

### 8. Generación de un resumen final

Al terminar la revisión, el bloque produce un resumen general con:

* Cantidad total de variables solicitadas.
* Cantidad de variables encontradas en `bd`.
* Cantidad de variables no encontradas.

Cuando existen variables faltantes, también se muestra una lista con sus nombres.

## Resultado producido

El bloque produce un reporte descriptivo en la salida del notebook para cada variable disponible.

Este reporte permite verificar:

* Si las columnas esperadas están presentes.
* Si los tipos de datos son adecuados.
* Si existen valores nulos.
* Cuántas categorías o valores diferentes contiene cada variable.
* Si alguna variable presenta una cantidad inesperada de valores únicos.
* Si las variables binarias y las variables creadas mediante One-Hot Encoding contienen valores coherentes.
* Si alguna columna necesaria para las etapas posteriores está ausente.

El bloque no modifica los valores del DataFrame. Su función es exclusivamente realizar una inspección estructural y descriptiva de las variables.


In [ ]:

# ============================================================
# VARIABLES INICIALES
# ============================================================

variables_iniciales = [
    "id_registro",
    "num_personas",
    "superficie_m2",
    "antiguedad_construccion_anios",
    "dias_facturacion",
    "temperatura_promedio_c",
    "tiene_aire_acondicionado",
    "cantidad_unidades_aa",
    "horas_uso_aa_dia",
    "tiene_calentador_agua_electrico",
    "tiene_lavadora",
    "cantidad_tv_o_pantallas",
    "cantidad_computadoras",
    "pct_iluminacion_led",
    "cantidad_focos",
    "horas_uso_iluminacion_dia",
    "otros_equipos_pequenos",
    "cantidad_equipos_total",
    "horas_dia_cocina",
    "horas_dia_sala_estar",
    "horas_dia_dormitorios",
    "horas_dia_oficina_estudio",
    "horas_dia_lavanderia",
    "antiguedad_electrodomesticos_anios",
    "dias_sin_electricidad_mes",
    "horas_uso_planta_o_inversor_mes",
    "generacion_solar_kwh_mensual",
    "certificacion_energetica_previa",
    "consumo_kwh_mes_anterior",
    "variacion_pct_consumo_mensual",
    "consumo_kwh_mensual",
    "consumo_neto_facturado_kwh",
    "costo_estimado_usd",
    "consumo_kwh_por_m2",
    "consumo_kwh_por_persona",
    "perfil_energetico",
    "personas_corregidas",
    "superficie_corregida",
    "aa_inconsistente",
    "iluminacion_inconsistente",
    "cantidad_equipos_reconstruida",
    "inicio_consumo_desde_cero",
    "sin_consumo_dos_meses",
    "caida_consumo_a_cero",
    "respaldo_inconsistente",
    "tipo_inmueble_Apartamento",
    "tipo_inmueble_Casa Unifamiliar",
    "tipo_inmueble_Pequeño Establecimiento Comercial",
    "zona_Suburbana",
    "zona_Urbana Costera",
    "zona_Urbana Interior",
    "nivel_socioeconomico_Alto",
    "nivel_socioeconomico_Bajo",
    "nivel_socioeconomico_Medio",
    "mes_referencia_Abril",
    "mes_referencia_Agosto",
    "mes_referencia_Diciembre",
    "mes_referencia_Enero",
    "mes_referencia_Febrero",
    "mes_referencia_Julio",
    "mes_referencia_Junio",
    "mes_referencia_Marzo",
    "mes_referencia_Mayo",
    "mes_referencia_Noviembre",
    "mes_referencia_Octubre",
    "mes_referencia_Septiembre",
    "horario_pico_uso_Madrugada",
    "horario_pico_uso_Mañana",
    "horario_pico_uso_Noche",
    "horario_pico_uso_Tarde",
    "aislamiento_termico_Bueno",
    "aislamiento_termico_Malo",
    "aislamiento_termico_Regular",
    "fuente_energia_secundaria_Inversor con baterías",
    "fuente_energia_secundaria_Ninguna",
    "fuente_energia_secundaria_Panel Solar",
    "fuente_energia_secundaria_Planta Eléctrica"
]


# ============================================================
# CANTIDAD MÁXIMA DE VALORES A MOSTRAR
# ============================================================

limite_general = 15

# Variables para las cuales conviene mostrar más categorías.
limites_especiales = {
    "perfil_energetico": 30,
    "certificacion_energetica_previa": 30
}


# ============================================================
# COMPROBAR COLUMNAS DISPONIBLES
# ============================================================

variables_disponibles = [
    variable
    for variable in variables_iniciales
    if variable in bd.columns
]

variables_faltantes = [
    variable
    for variable in variables_iniciales
    if variable not in bd.columns
]


# ============================================================
# RECORRER VARIABLES Y MOSTRAR VALORES ÚNICOS
# ============================================================

for numero, variable in enumerate(variables_disponibles, start=1):

    serie = bd[variable]

    limite = limites_especiales.get(
        variable,
        limite_general
    )

    cantidad_unicos_sin_nulos = serie.nunique(dropna=True)
    cantidad_unicos_con_nulos = serie.nunique(dropna=False)
    cantidad_nulos = serie.isna().sum()

    valores_unicos = serie.drop_duplicates()

    # Intenta ordenar los valores.
    try:
        valores_unicos = valores_unicos.sort_values(
            na_position="last"
        )
    except (TypeError, ValueError):
        # Puede fallar cuando hay tipos de datos mezclados.
        pass

    valores_mostrados = valores_unicos.head(limite).tolist()

    print("=" * 80)
    print(f"{numero}. VARIABLE: {variable}")
    print(f"Tipo de dato: {serie.dtype}")
    print(
        f"Valores únicos sin contar nulos: "
        f"{cantidad_unicos_sin_nulos}"
    )
    print(
        f"Valores únicos contando nulos: "
        f"{cantidad_unicos_con_nulos}"
    )
    print(f"Cantidad de valores nulos: {cantidad_nulos}")
    print(f"Límite de valores mostrados: {limite}")
    print(f"Valores únicos mostrados: {valores_mostrados}")

    if cantidad_unicos_con_nulos > limite:
        print(
            f"Se muestran solamente los primeros {limite} "
            f"de {cantidad_unicos_con_nulos} valores únicos."
        )


# ============================================================
# RESUMEN FINAL
# ============================================================

print("\n" + "=" * 80)
print("RESUMEN")
print("=" * 80)
print(f"Variables solicitadas: {len(variables_iniciales)}")
print(f"Variables encontradas en bd: {len(variables_disponibles)}")
print(f"Variables no encontradas: {len(variables_faltantes)}")

if variables_faltantes:
    print("\nVariables faltantes:")
    for variable in variables_faltantes:
        print(f"- {variable}")

1. VARIABLE: id_registro
Tipo de dato: object
Valores únicos sin contar nulos: 94537
Valores únicos contando nulos: 94537
Cantidad de valores nulos: 0
Límite de valores mostrados: 15
Valores únicos mostrados: ['  REG-000106  ', '  REG-000205  ', '  REG-000408  ', '  REG-000502  ', '  REG-000535  ', '  REG-000649  ', '  REG-000736  ', '  REG-000848  ', '  REG-001008  ', '  REG-001149  ', '  REG-001227  ', '  REG-001259  ', '  REG-001270  ', '  REG-001367  ', '  REG-001652  ']
Se muestran solamente los primeros 15 de 94537 valores únicos.
2. VARIABLE: num_personas
Tipo de dato: int64
Valores únicos sin contar nulos: 14
Valores únicos contando nulos: 14
Cantidad de valores nulos: 0
Límite de valores mostrados: 15
Valores únicos mostrados: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 50, 88]
3. VARIABLE: superficie_m2
Tipo de dato: float64
Valores únicos sin contar nulos: 10015
Valores únicos contando nulos: 10015
Cantidad de valores nulos: 0
Límite de valores mostrados: 15
Valores únicos mostr

### Preparación de variables auxiliares y creación de características

Este bloque tiene como propósito preparar las variables necesarias para el proceso de ingeniería de características, construir nuevas variables explicativas y agregarlas al DataFrame `bd`.

El procedimiento transforma las columnas originales en representaciones numéricas consistentes, genera indicadores asociados al consumo energético y valida que las nuevas variables no introduzcan valores inválidos. También conserva un registro de las columnas existentes antes de la transformación para producir un resumen final de los cambios realizados.

## Funcionalidades que cumple

### 1. Registro de las variables originales

El bloque guarda los nombres de las columnas que existen inicialmente en `bd`.

Este registro se utiliza al final para distinguir entre:

* Variables completamente nuevas.
* Variables que ya existían y fueron reemplazadas.
* Cantidad inicial y final de columnas.

### 2. Preparación de variables numéricas

Las variables utilizadas en los cálculos se convierten a formato numérico.

Los valores que no pueden convertirse correctamente se consideran faltantes y posteriormente se reemplazan por cero. Esta preparación se aplica a variables relacionadas con:

* Personas y superficie.
* Antigüedad del inmueble.
* Días de facturación.
* Temperatura.
* Aire acondicionado.
* Equipamiento.
* Iluminación.
* Cortes eléctricos.
* Respaldo energético.
* Generación solar.
* Consumo histórico.
* Certificación energética.

El porcentaje de iluminación LED se restringe al intervalo comprendido entre 0 y 100 y se transforma a una proporción entre 0 y 1.

La variación porcentual del consumo también se ajusta cuando sus valores parecen estar expresados como porcentaje completo, convirtiéndolos a una proporción decimal.

### 3. Preparación de las horas de uso de los espacios

Se agrupan las horas de uso correspondientes a:

* Cocina.
* Sala de estar.
* Dormitorios.
* Oficina o estudio.
* Lavandería.

A partir de estas variables se producen indicadores auxiliares como:

* Horas totales de uso de los espacios.
* Promedio de horas de uso.
* Cantidad de espacios activos.
* Índice de permanencia en el hogar.
* Índice de actividad doméstica.

Estos indicadores resumen el nivel y la distribución de actividad dentro del inmueble.

### 4. Preparación de variables de climatización

El bloque calcula los grados de calor por encima de una temperatura de referencia de 24 °C.

También produce:

* Carga operativa del aire acondicionado.
* Factor de aislamiento térmico.
* Demanda térmica estimada.

El factor de aislamiento asigna una ponderación diferente a los inmuebles con aislamiento malo, regular o bueno. Cuando no se identifica una categoría válida, se utiliza un factor neutral.

Estas variables permiten representar conjuntamente la temperatura, la superficie, el aislamiento y el uso del aire acondicionado.

### 5. Preparación de variables de equipamiento

Se recuperan y normalizan las cantidades de televisores, computadoras, equipos pequeños, lavadoras, calentadores eléctricos y unidades de aire acondicionado.

A partir de ellas se producen:

* Carga tecnológica.
* Cantidad de equipos de alto consumo.

La carga tecnológica asigna diferentes ponderaciones a los tipos de equipos para representar su importancia relativa dentro del consumo energético.

### 6. Preparación de variables de iluminación

Se estima la cantidad de focos LED y no LED utilizando la cantidad total de focos y la proporción de iluminación LED.

Posteriormente se producen:

* Cantidad de focos LED.
* Cantidad de focos no LED.
* Carga de iluminación LED.
* Carga de iluminación no LED.
* Carga total de iluminación.

Estas variables combinan la cantidad de focos, su tecnología y las horas de uso.

### 7. Preparación de variables solares

Se calcula la relación entre la generación solar mensual y el consumo del mes anterior.

También se obtiene la diferencia entre ambas cantidades.

Como resultado se producen dos indicadores auxiliares:

* Tasa previa de autogeneración.
* Brecha energética previa.

### 8. Preparación de variables de cortes y respaldo

Se generan indicadores relacionados con la interrupción del suministro eléctrico y la capacidad de respaldo del inmueble.

Entre ellos se encuentran:

* Frecuencia relativa de cortes.
* Cobertura estimada del respaldo.
* Índice de vulnerabilidad eléctrica.

También se recuperan las variables binarias que identifican si la fuente secundaria corresponde a:

* Inversor con baterías.
* Ausencia de respaldo.
* Panel solar.
* Planta eléctrica.

El índice de vulnerabilidad combina la frecuencia de los cortes, la cobertura disponible y la ausencia de una fuente secundaria.

### 9. Preparación de variables categóricas codificadas

El bloque recupera como variables numéricas las columnas generadas mediante One-Hot Encoding para representar:

* Tipo de inmueble.
* Zona.
* Nivel socioeconómico.
* Horario pico de uso.
* Aislamiento térmico.
* Fuente de energía secundaria.

Estas variables se utilizan posteriormente para crear interacciones entre las características físicas, energéticas y categóricas del inmueble.

### 10. Reconstrucción del mes de referencia

Las doce columnas binarias correspondientes a los meses se convierten en un número de mes comprendido entre 1 y 12.

Después se generan dos representaciones cíclicas del mes.

Estas representaciones permiten conservar la relación circular del calendario, de modo que diciembre y enero puedan considerarse meses cercanos en lugar de extremos opuestos.

### 11. Recuperación de banderas de calidad

El bloque reúne las banderas generadas durante las fases anteriores para identificar:

* Variables que originalmente eran nulas.
* Correcciones determinísticas.
* Inconsistencias.
* Reconstrucciones.
* Patrones especiales de consumo.

Cuando una bandera esperada no existe en `bd`, se crea temporalmente una serie de ceros para mantener una estructura uniforme en los cálculos posteriores.

Todas las banderas se convierten a valores enteros.

### 12. Comprobación de variables auxiliares

Se construye una lista con todos los alias e indicadores auxiliares preparados.

Al finalizar esta parte, se muestra la cantidad total de variables auxiliares creadas correctamente.

### 13. Detección de divisiones no calculables

El bloque identifica las filas en las que alguno de los denominadores utilizados en las nuevas características es igual a cero.

Se revisan denominadores como:

* Número de personas.
* Superficie.
* Cantidad de equipos.
* Horas totales de actividad.
* Antigüedad del inmueble.
* Días de facturación.
* Consumo anterior.
* Días sin electricidad.

Para cada denominador se crea una bandera únicamente cuando existe al menos un registro en el que la división no puede calcularse.

Las banderas toman los siguientes significados:

* Valor 1: el denominador es igual a cero.
* Valor 0: la división puede realizarse.

Si no existen casos problemáticos, la bandera correspondiente no se agrega al DataFrame.

## Variables que produce

### 14. Variables de ocupación y superficie

El bloque crea indicadores que relacionan:

* Superficie y número de personas.
* Personas y superficie.
* Superficie y cantidad de equipos.
* Personas y cantidad de equipos.

Estas variables representan la densidad de ocupación y la disponibilidad de espacio o equipamiento por residente.

### 15. Variables de uso de espacios

Se producen características relacionadas con:

* Uso total y promedio de los espacios.
* Uso por persona.
* Uso por metro cuadrado.
* Cantidad de espacios activos.
* Proporción de uso de cada habitación.
* Nivel de teletrabajo.
* Actividad doméstica.
* Permanencia estimada en el hogar.

### 16. Variables de aire acondicionado

Se crean características que combinan:

* Temperatura.
* Grados de calor.
* Cantidad de unidades.
* Horas de uso.
* Personas.
* Superficie.
* Tipo de aislamiento.

Estas variables representan la intensidad de climatización y la demanda térmica del inmueble.

### 17. Variables de equipamiento

Se generan indicadores como:

* Equipos por persona.
* Equipos por metro cuadrado.
* Carga tecnológica.
* Equipos de alto consumo.
* Carga asociada a equipos antiguos.
* Índice de obsolescencia.
* Diferencia de antigüedad entre el inmueble y sus equipos.

### 18. Variables de iluminación

Se producen indicadores relacionados con:

* Proporción de focos LED.
* Cantidad de focos LED y no LED.
* Focos por persona.
* Focos por metro cuadrado.
* Carga de iluminación por persona.
* Carga de iluminación por superficie.
* Ineficiencia estimada de la iluminación.

### 19. Variables de consumo histórico

Se crean medidas del consumo anterior ajustadas por:

* Día de facturación.
* Persona.
* Superficie.
* Equipo.
* Hora de actividad.

También se generan variables que representan:

* Consumo mensual ajustado a 30 días.
* Cambio estimado del consumo.
* Consumo esperado según la tendencia.
* Magnitud de la variación.

### 20. Variables de energía solar

Se producen características relacionadas con:

* Generación solar diaria.
* Generación por persona.
* Generación por superficie.
* Tasa de autogeneración.
* Brecha entre generación y consumo.
* Dependencia estimada de la red.
* Existencia de generación solar.
* Intensidad de generación.

### 21. Variables de cortes y respaldo

Se generan variables que representan:

* Intensidad de los cortes.
* Uso del respaldo por día de corte.
* Uso diario del respaldo.
* Dependencia del sistema de respaldo.
* Cobertura estimada.
* Cortes asociados con cada tipo de fuente secundaria.
* Vulnerabilidad eléctrica.

### 22. Interacciones con el tipo de inmueble

Las variables de apartamento, casa y establecimiento comercial se combinan con:

* Superficie.
* Ocupación.
* Cantidad de equipos.
* Horas de actividad.
* Consumo histórico.

Estas interacciones permiten que el modelo diferencie los patrones energéticos según el tipo de inmueble.

### 23. Variables de aislamiento y antigüedad

Se generan características que relacionan:

* Antigüedad del inmueble.
* Calidad del aislamiento.
* Demanda térmica.
* Antigüedad de los electrodomésticos.

También se crean indicadores binarios para identificar:

* Inmuebles con 30 años o más.
* Electrodomésticos con 10 años o más.
* Registros donde tanto el inmueble como los equipos son antiguos.

### 24. Variables de estacionalidad

Se crean características que combinan el mes con:

* Temperatura.
* Uso del aire acondicionado.
* Generación solar.
* Consumo anterior.

Estas variables permiten representar cambios estacionales en los patrones de consumo.

### 25. Variables asociadas al horario pico

Se agrupan los horarios en:

* Uso diurno.
* Uso nocturno.

También se generan interacciones entre el horario pico y:

* Aire acondicionado.
* Iluminación.
* Actividad en los espacios.

### 26. Interacciones con la zona

Las categorías de zona se combinan con:

* Temperatura.
* Uso del aire acondicionado.
* Generación solar.
* Vulnerabilidad eléctrica.

Esto permite capturar diferencias energéticas entre zonas suburbanas, costeras e interiores.

### 27. Interacciones con el nivel socioeconómico

El nivel socioeconómico se combina con:

* Cantidad de equipos.
* Uso del aire acondicionado.
* Superficie.
* Consumo anterior.

Estas variables permiten representar patrones diferentes entre los niveles alto, medio y bajo.

### 28. Interacciones con la certificación energética

La certificación energética se relaciona con:

* Antigüedad del inmueble.
* Aislamiento térmico.
* Proporción de iluminación LED.
* Consumo histórico.

## Variables derivadas de las banderas

### 29. Conteos de calidad del registro

Las banderas existentes y las banderas de divisiones no calculables se combinan para producir:

* Cantidad total de banderas activas.
* Cantidad de variables que fueron imputadas.
* Cantidad de correcciones determinísticas.
* Cantidad de inconsistencias.
* Cantidad de anomalías relacionadas con el consumo.

### 30. Indicadores generales de calidad

También se generan variables binarias que indican si el registro:

* Presenta alguna inconsistencia.
* Contiene alguna variable imputada.
* Recibió alguna corrección.
* Presenta una inconsistencia operativa.
* Presenta una inconsistencia de consumo.
* Tiene un patrón de consumo igual a cero.
* Presenta un cambio extremo del consumo.
* Tiene un consumo histórico anómalo.
* Contiene información de consumo potencialmente no confiable.

### 31. Índice de calidad del registro

Se calcula un índice entre 0 y 1 a partir de la proporción de banderas activas.

Un valor más cercano a 1 representa un registro con menos incidencias de calidad, mientras que un valor menor indica una mayor cantidad de correcciones, imputaciones, inconsistencias o divisiones no calculables.

## Incorporación y validación de resultados

### 32. Construcción del DataFrame de características

Todas las variables generadas se reúnen en un nuevo DataFrame con el mismo índice de `bd`.

Los valores infinitos se sustituyen por valores nulos para facilitar su identificación durante la auditoría.

### 33. Auditoría de valores nulos

Antes de incorporar las variables, se comprueba cuáles contienen valores faltantes.

Cuando se encuentran nulos, se muestra:

* Nombre de la variable.
* Cantidad de valores nulos.
* Total general de nulos en las variables derivadas.

Si no existen valores faltantes, se informa que las características fueron generadas sin nulos.

### 34. Prevención de columnas duplicadas

Las variables nuevas que tengan el mismo nombre que una columna existente se identifican como variables reemplazadas.

Las versiones anteriores se eliminan antes de incorporar los nuevos resultados, evitando que `bd` contenga columnas duplicadas.

### 35. Incorporación de las nuevas variables

El DataFrame de características se concatena con `bd` utilizando el índice de los registros.

Como resultado, la base original queda ampliada con todas las variables derivadas.

### 36. Normalización de las banderas de división

Las banderas creadas para divisiones no calculables se convierten a valores enteros.

También se comprueba que:

* No contengan valores nulos.
* Solo presenten los valores 0 y 1.

Si alguna bandera incumple estas condiciones, la ejecución se detiene.

### 37. Resumen del proceso

El bloque produce un reporte final con:

* Cantidad de filas procesadas.
* Número de variables iniciales.
* Número de variables calculadas.
* Cantidad de variables nuevas.
* Cantidad de variables reemplazadas.
* Número de banderas de división creadas.
* Total final de variables.

Cuando existen banderas de división, también se muestra para cada una:

* Cantidad de casos detectados.
* Porcentaje que representan dentro del conjunto de datos.

### 38. Validaciones finales

Finalmente se comprueba:

* Cantidad de valores infinitos.
* Cantidad de valores nulos.
* Número de variables nuevas constantes.
* Nombres de las variables constantes.
* Columnas que todavía contienen valores faltantes.

## Resultado producido

El bloque produce una versión ampliada del DataFrame `bd` que contiene:

* Las variables originales.
* Nuevas características físicas y operativas.
* Relaciones entre ocupación, superficie y equipamiento.
* Indicadores de climatización e iluminación.
* Características de consumo histórico.
* Variables solares y de respaldo eléctrico.
* Interacciones con categorías del inmueble.
* Indicadores estacionales.
* Variables de calidad y confiabilidad del registro.
* Banderas para identificar divisiones no calculables.

El resultado final queda preparado para procesos posteriores de selección de variables, entrenamiento de modelos y evaluación predictiva.


In [ ]:
# ============================================================
# ALIAS Y VARIABLES AUXILIARES
# ============================================================

# Guardar las columnas originales para el resumen final.
columnas_iniciales = bd.columns.tolist()


# ============================================================
# ALIAS DE VARIABLES NUMÉRICAS
# ============================================================

num_personas = (
    pd.to_numeric(
        bd["num_personas"],
        errors="coerce"
    )
    .fillna(0)
)

superficie_m2 = (
    pd.to_numeric(
        bd["superficie_m2"],
        errors="coerce"
    )
    .fillna(0)
)

antiguedad_inmueble = (
    pd.to_numeric(
        bd["antiguedad_construccion_anios"],
        errors="coerce"
    )
    .fillna(0)
)

dias_facturacion = (
    pd.to_numeric(
        bd["dias_facturacion"],
        errors="coerce"
    )
    .fillna(0)
)

temperatura = (
    pd.to_numeric(
        bd["temperatura_promedio_c"],
        errors="coerce"
    )
    .fillna(0)
)

cantidad_unidades_aa = (
    pd.to_numeric(
        bd["cantidad_unidades_aa"],
        errors="coerce"
    )
    .fillna(0)
)

horas_uso_aa = (
    pd.to_numeric(
        bd["horas_uso_aa_dia"],
        errors="coerce"
    )
    .fillna(0)
)

cantidad_equipos = (
    pd.to_numeric(
        bd["cantidad_equipos_total"],
        errors="coerce"
    )
    .fillna(0)
)

cantidad_focos = (
    pd.to_numeric(
        bd["cantidad_focos"],
        errors="coerce"
    )
    .fillna(0)
)

horas_iluminacion = (
    pd.to_numeric(
        bd["horas_uso_iluminacion_dia"],
        errors="coerce"
    )
    .fillna(0)
)

porcentaje_led = (
    pd.to_numeric(
        bd["pct_iluminacion_led"],
        errors="coerce"
    )
    .fillna(0)
    .clip(lower=0, upper=100)
    / 100
)

antiguedad_equipos = (
    pd.to_numeric(
        bd["antiguedad_electrodomesticos_anios"],
        errors="coerce"
    )
    .fillna(0)
)

dias_sin_electricidad = (
    pd.to_numeric(
        bd["dias_sin_electricidad_mes"],
        errors="coerce"
    )
    .fillna(0)
)

horas_respaldo = (
    pd.to_numeric(
        bd["horas_uso_planta_o_inversor_mes"],
        errors="coerce"
    )
    .fillna(0)
)

generacion_solar = (
    pd.to_numeric(
        bd["generacion_solar_kwh_mensual"],
        errors="coerce"
    )
    .fillna(0)
)

consumo_anterior = (
    pd.to_numeric(
        bd["consumo_kwh_mes_anterior"],
        errors="coerce"
    )
    .fillna(0)
)

variacion_consumo = (
    pd.to_numeric(
        bd["variacion_pct_consumo_mensual"],
        errors="coerce"
    )
    .fillna(0)
)

if variacion_consumo.abs().max() > 2:
    variacion_consumo = variacion_consumo / 100

certificacion = (
    pd.to_numeric(
        bd["certificacion_energetica_previa"],
        errors="coerce"
    )
    .fillna(0)
)


# ============================================================
# USO DE ESPACIOS
# ============================================================

columnas_horas_espacios = [
    "horas_dia_cocina",
    "horas_dia_sala_estar",
    "horas_dia_dormitorios",
    "horas_dia_oficina_estudio",
    "horas_dia_lavanderia"
]

horas_espacios = (
    bd[columnas_horas_espacios]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .fillna(0)
)

horas_uso_espacios_total = (
    horas_espacios.sum(axis=1)
)

horas_uso_espacios_promedio = (
    horas_espacios.mean(axis=1)
)

cantidad_espacios_activos = (
    horas_espacios
    .gt(0)
    .sum(axis=1)
    .astype(np.int64)
)

indice_permanencia_hogar = (
    horas_uso_espacios_total
    / (
        24
        * len(columnas_horas_espacios)
    )
).clip(lower=0, upper=1)

indice_actividad_domestica = (
    horas_espacios[
        [
            "horas_dia_cocina",
            "horas_dia_lavanderia"
        ]
    ]
    .sum(axis=1)
)


# ============================================================
# AIRE ACONDICIONADO Y TEMPERATURA
# ============================================================

grados_calor = (
    temperatura - 24
).clip(lower=0)

carga_operativa_aa = (
    cantidad_unidades_aa
    * horas_uso_aa
)

aislamiento_malo = (
    pd.to_numeric(
        bd["aislamiento_termico_Malo"],
        errors="coerce"
    )
    .fillna(0)
)

aislamiento_regular = (
    pd.to_numeric(
        bd["aislamiento_termico_Regular"],
        errors="coerce"
    )
    .fillna(0)
)

aislamiento_bueno = (
    pd.to_numeric(
        bd["aislamiento_termico_Bueno"],
        errors="coerce"
    )
    .fillna(0)
)

factor_aislamiento = (
    aislamiento_malo * 1.30
    + aislamiento_regular * 1.00
    + aislamiento_bueno * 0.70
)

factor_aislamiento = (
    factor_aislamiento.where(
        factor_aislamiento.gt(0),
        1
    )
)

demanda_termica_estimada = (
    grados_calor
    * superficie_m2
    * factor_aislamiento
)


# ============================================================
# EQUIPAMIENTO
# ============================================================

cantidad_tv = (
    pd.to_numeric(
        bd["cantidad_tv_o_pantallas"],
        errors="coerce"
    )
    .fillna(0)
)

cantidad_computadoras = (
    pd.to_numeric(
        bd["cantidad_computadoras"],
        errors="coerce"
    )
    .fillna(0)
)

otros_equipos = (
    pd.to_numeric(
        bd["otros_equipos_pequenos"],
        errors="coerce"
    )
    .fillna(0)
)

tiene_lavadora = (
    pd.to_numeric(
        bd["tiene_lavadora"],
        errors="coerce"
    )
    .fillna(0)
)

tiene_calentador = (
    pd.to_numeric(
        bd["tiene_calentador_agua_electrico"],
        errors="coerce"
    )
    .fillna(0)
)

carga_tecnologica = (
    cantidad_tv
    + cantidad_computadoras * 1.5
    + otros_equipos * 0.5
)

cantidad_equipos_alto_consumo = (
    cantidad_unidades_aa
    + tiene_lavadora
    + tiene_calentador
)


# ============================================================
# ILUMINACIÓN
# ============================================================

cantidad_focos_led = (
    cantidad_focos
    * porcentaje_led
)

cantidad_focos_no_led = (
    cantidad_focos
    - cantidad_focos_led
)

carga_iluminacion_led = (
    cantidad_focos_led
    * horas_iluminacion
)

carga_iluminacion_no_led = (
    cantidad_focos_no_led
    * horas_iluminacion
)

carga_iluminacion_total = (
    carga_iluminacion_led
    + carga_iluminacion_no_led
)


# ============================================================
# ENERGÍA SOLAR
# ============================================================

tasa_autogeneracion_previa = division_segura(
    generacion_solar,
    consumo_anterior,
    valor_por_defecto=0
).clip(lower=0)

brecha_energetica_previa = (
    generacion_solar
    - consumo_anterior
)


# ============================================================
# CORTES Y RESPALDO
# ============================================================

frecuencia_cortes_relativa = division_segura(
    dias_sin_electricidad,
    dias_facturacion,
    valor_por_defecto=0
).clip(lower=0, upper=1)

cobertura_respaldo_estimada = division_segura(
    horas_respaldo,
    dias_sin_electricidad * 24,
    valor_por_defecto=0
).clip(lower=0, upper=1)

respaldo_inversor = (
    pd.to_numeric(
        bd["fuente_energia_secundaria_Inversor con baterías"],
        errors="coerce"
    )
    .fillna(0)
)

respaldo_ninguno = (
    pd.to_numeric(
        bd["fuente_energia_secundaria_Ninguna"],
        errors="coerce"
    )
    .fillna(0)
)

respaldo_solar = (
    pd.to_numeric(
        bd["fuente_energia_secundaria_Panel Solar"],
        errors="coerce"
    )
    .fillna(0)
)

respaldo_planta = (
    pd.to_numeric(
        bd["fuente_energia_secundaria_Planta Eléctrica"],
        errors="coerce"
    )
    .fillna(0)
)

indice_vulnerabilidad_electrica = (
    frecuencia_cortes_relativa
    * (1 - cobertura_respaldo_estimada)
    * respaldo_ninguno.add(1)
)


# ============================================================
# TIPO DE INMUEBLE
# ============================================================

apartamento = (
    pd.to_numeric(
        bd["tipo_inmueble_Apartamento"],
        errors="coerce"
    )
    .fillna(0)
)

casa = (
    pd.to_numeric(
        bd["tipo_inmueble_Casa Unifamiliar"],
        errors="coerce"
    )
    .fillna(0)
)

comercial = (
    pd.to_numeric(
        bd["tipo_inmueble_Pequeño Establecimiento Comercial"],
        errors="coerce"
    )
    .fillna(0)
)


# ============================================================
# ZONA
# ============================================================

zona_suburbana = (
    pd.to_numeric(
        bd["zona_Suburbana"],
        errors="coerce"
    )
    .fillna(0)
)

zona_costera = (
    pd.to_numeric(
        bd["zona_Urbana Costera"],
        errors="coerce"
    )
    .fillna(0)
)

zona_interior = (
    pd.to_numeric(
        bd["zona_Urbana Interior"],
        errors="coerce"
    )
    .fillna(0)
)


# ============================================================
# NIVEL SOCIOECONÓMICO
# ============================================================

nivel_alto = (
    pd.to_numeric(
        bd["nivel_socioeconomico_Alto"],
        errors="coerce"
    )
    .fillna(0)
)

nivel_medio = (
    pd.to_numeric(
        bd["nivel_socioeconomico_Medio"],
        errors="coerce"
    )
    .fillna(0)
)

nivel_bajo = (
    pd.to_numeric(
        bd["nivel_socioeconomico_Bajo"],
        errors="coerce"
    )
    .fillna(0)
)


# ============================================================
# HORARIO PICO
# ============================================================

pico_madrugada = (
    pd.to_numeric(
        bd["horario_pico_uso_Madrugada"],
        errors="coerce"
    )
    .fillna(0)
)

pico_manana = (
    pd.to_numeric(
        bd["horario_pico_uso_Mañana"],
        errors="coerce"
    )
    .fillna(0)
)

pico_noche = (
    pd.to_numeric(
        bd["horario_pico_uso_Noche"],
        errors="coerce"
    )
    .fillna(0)
)

pico_tarde = (
    pd.to_numeric(
        bd["horario_pico_uso_Tarde"],
        errors="coerce"
    )
    .fillna(0)
)


# ============================================================
# MES DE REFERENCIA
# ============================================================

columnas_meses = {
    1: "mes_referencia_Enero",
    2: "mes_referencia_Febrero",
    3: "mes_referencia_Marzo",
    4: "mes_referencia_Abril",
    5: "mes_referencia_Mayo",
    6: "mes_referencia_Junio",
    7: "mes_referencia_Julio",
    8: "mes_referencia_Agosto",
    9: "mes_referencia_Septiembre",
    10: "mes_referencia_Octubre",
    11: "mes_referencia_Noviembre",
    12: "mes_referencia_Diciembre"
}

mes_numero = pd.Series(
    0,
    index=bd.index,
    dtype=np.int64
)

for numero_mes, columna_mes in columnas_meses.items():

    if columna_mes in bd.columns:

        mes_numero = mes_numero.mask(
            pd.to_numeric(
                bd[columna_mes],
                errors="coerce"
            )
            .fillna(0)
            .eq(1),
            numero_mes
        )

mes_sin = np.sin(
    2 * np.pi * mes_numero / 12
)

mes_cos = np.cos(
    2 * np.pi * mes_numero / 12
)


# ============================================================
# BANDERAS DE CALIDAD EXISTENTES
# ============================================================

nombres_banderas_calidad = [
    "num_personas_era_nulo",
    "superficie_m2_era_nulo",
    "dias_facturacion_era_nulo",
    "cantidad_unidades_aa_era_nulo",
    "horas_uso_aa_dia_era_nulo",
    "cantidad_focos_era_nulo",
    "pct_iluminacion_led_era_nulo",
    "horas_uso_iluminacion_dia_era_nulo",
    "tiene_lavadora_era_nulo",
    "horas_dia_lavanderia_era_nulo",
    "generacion_solar_kwh_mensual_era_nulo",
    "consumo_kwh_mes_anterior_era_nulo",
    "registro_con_correccion_deterministica",
    "personas_corregidas",
    "superficie_corregida",
    "cantidad_equipos_reconstruida",
    "equipos_total_inconsistente",
    "aa_inconsistente",
    "iluminacion_inconsistente",
    "respaldo_inconsistente",
    "inicio_consumo_desde_cero",
    "sin_consumo_dos_meses",
    "caida_consumo_a_cero"
]

banderas_calidad = {}

for nombre_bandera in nombres_banderas_calidad:

    if nombre_bandera in bd.columns:

        banderas_calidad[nombre_bandera] = (
            pd.to_numeric(
                bd[nombre_bandera],
                errors="coerce"
            )
            .fillna(0)
            .astype(np.int64)
        )

    else:

        banderas_calidad[nombre_bandera] = pd.Series(
            0,
            index=bd.index,
            dtype=np.int64
        )


# ============================================================
# COMPROBACIÓN FINAL DE ALIAS
# ============================================================

alias_creados = [
    "num_personas",
    "superficie_m2",
    "antiguedad_inmueble",
    "dias_facturacion",
    "temperatura",
    "cantidad_unidades_aa",
    "horas_uso_aa",
    "cantidad_equipos",
    "cantidad_focos",
    "horas_iluminacion",
    "porcentaje_led",
    "antiguedad_equipos",
    "dias_sin_electricidad",
    "horas_respaldo",
    "generacion_solar",
    "consumo_anterior",
    "variacion_consumo",
    "certificacion",
    "horas_uso_espacios_total",
    "horas_uso_espacios_promedio",
    "cantidad_espacios_activos",
    "indice_permanencia_hogar",
    "indice_actividad_domestica",
    "grados_calor",
    "carga_operativa_aa",
    "factor_aislamiento",
    "demanda_termica_estimada",
    "carga_tecnologica",
    "cantidad_equipos_alto_consumo",
    "cantidad_focos_led",
    "cantidad_focos_no_led",
    "carga_iluminacion_led",
    "carga_iluminacion_no_led",
    "carga_iluminacion_total",
    "tasa_autogeneracion_previa",
    "brecha_energetica_previa",
    "frecuencia_cortes_relativa",
    "cobertura_respaldo_estimada",
    "indice_vulnerabilidad_electrica",
    "apartamento",
    "casa",
    "comercial",
    "zona_suburbana",
    "zona_costera",
    "zona_interior",
    "nivel_alto",
    "nivel_medio",
    "nivel_bajo",
    "pico_madrugada",
    "pico_manana",
    "pico_noche",
    "pico_tarde",
    "mes_numero",
    "mes_sin",
    "mes_cos"
]

print(
    f"Se crearon correctamente "
    f"{len(alias_creados)} alias y variables auxiliares."
)


# ============================================================
# DIVISIONES NO CALCULABLES
# ============================================================

# Se revisan todos los denominadores variables utilizados por
# las nuevas características.
#
# La bandera tendrá:
#   1 = el denominador es igual a cero
#   0 = la división puede realizarse
#
# IMPORTANTE:
# La columna bandera se crea únicamente cuando existe al menos
# una fila con denominador igual a cero. Si no existen casos,
# la bandera no se agrega al DataFrame.

mascaras_division_no_calculable = {
    "division_por_personas_no_calculable": (
        num_personas.eq(0)
    ),

    "division_por_superficie_no_calculable": (
        superficie_m2.eq(0)
    ),

    "division_por_equipos_no_calculable": (
        cantidad_equipos.eq(0)
    ),

    "division_por_horas_actividad_no_calculable": (
        horas_uso_espacios_total.eq(0)
    ),

    "division_por_antiguedad_inmueble_no_calculable": (
        antiguedad_inmueble.eq(0)
    ),

    "division_por_dias_facturacion_no_calculable": (
        dias_facturacion.eq(0)
    ),

    "division_por_consumo_anterior_no_calculable": (
        consumo_anterior.eq(0)
    ),

    "division_por_dias_sin_electricidad_no_calculable": (
        dias_sin_electricidad.eq(0)
    )
}

banderas_division_no_calculable = {}

for nombre_bandera, mascara in (
    mascaras_division_no_calculable.items()
):
    mascara = mascara.fillna(False)

    # Crear la columna solamente si se detecta al menos
    # una división cuyo denominador sea igual a cero.
    if mascara.any():
        banderas_division_no_calculable[nombre_bandera] = (
            mascara.astype(np.int64)
        )


# ============================================================
# CREACIÓN DE FEATURES
# ============================================================

features = {

    # --------------------------------------------------------
    # OCUPACIÓN Y SUPERFICIE
    # --------------------------------------------------------

    "superficie_por_persona": division_segura(
        superficie_m2,
        num_personas,
        valor_por_defecto=0
    ),

    "densidad_habitacional": division_segura(
        num_personas,
        superficie_m2,
        valor_por_defecto=0
    ),

    "superficie_por_equipo": division_segura(
        superficie_m2,
        cantidad_equipos,
        valor_por_defecto=0
    ),

    "personas_por_equipo": division_segura(
        num_personas,
        cantidad_equipos,
        valor_por_defecto=0
    ),

    # --------------------------------------------------------
    # USO DE ESPACIOS
    # --------------------------------------------------------

    "horas_uso_espacios_total":
        horas_uso_espacios_total,

    "horas_uso_espacios_promedio":
        horas_uso_espacios_promedio,

    "uso_espacios_por_persona": division_segura(
        horas_uso_espacios_total,
        num_personas,
        valor_por_defecto=0
    ),

    "uso_espacios_por_m2": division_segura(
        horas_uso_espacios_total,
        superficie_m2,
        valor_por_defecto=0
    ),

    "cantidad_espacios_activos":
        cantidad_espacios_activos,

    "indice_permanencia_hogar":
        indice_permanencia_hogar,

    "proporcion_uso_cocina": division_segura(
        horas_espacios["horas_dia_cocina"],
        horas_uso_espacios_total,
        valor_por_defecto=0
    ),

    "proporcion_uso_sala": division_segura(
        horas_espacios["horas_dia_sala_estar"],
        horas_uso_espacios_total,
        valor_por_defecto=0
    ),

    "proporcion_uso_dormitorios": division_segura(
        horas_espacios["horas_dia_dormitorios"],
        horas_uso_espacios_total,
        valor_por_defecto=0
    ),

    "proporcion_uso_oficina": division_segura(
        horas_espacios["horas_dia_oficina_estudio"],
        horas_uso_espacios_total,
        valor_por_defecto=0
    ),

    "proporcion_uso_lavanderia": division_segura(
        horas_espacios["horas_dia_lavanderia"],
        horas_uso_espacios_total,
        valor_por_defecto=0
    ),

    "indice_teletrabajo": division_segura(
        horas_espacios["horas_dia_oficina_estudio"],
        24,
        valor_por_defecto=0
    ),

    "indice_actividad_domestica":
        indice_actividad_domestica,

    # --------------------------------------------------------
    # AIRE ACONDICIONADO
    # --------------------------------------------------------

    "grados_calor":
        grados_calor,

    "carga_operativa_aa":
        carga_operativa_aa,

    "carga_climatica_aa":
        carga_operativa_aa * grados_calor,

    "unidades_aa_por_persona": division_segura(
        cantidad_unidades_aa,
        num_personas,
        valor_por_defecto=0
    ),

    "unidades_aa_por_m2": division_segura(
        cantidad_unidades_aa,
        superficie_m2,
        valor_por_defecto=0
    ),

    "horas_aa_por_persona": division_segura(
        horas_uso_aa,
        num_personas,
        valor_por_defecto=0
    ),

    "intensidad_aa_por_persona": division_segura(
        carga_operativa_aa,
        num_personas,
        valor_por_defecto=0
    ),

    "intensidad_aa_por_m2": division_segura(
        carga_operativa_aa,
        superficie_m2,
        valor_por_defecto=0
    ),

    "aa_ajustado_por_aislamiento":
        carga_operativa_aa * factor_aislamiento,

    "demanda_termica_estimada":
        demanda_termica_estimada,

    "exposicion_termica":
        grados_calor * superficie_m2,

    "temperatura_aislamiento_malo":
        temperatura * aislamiento_malo,

    "temperatura_aislamiento_regular":
        temperatura * aislamiento_regular,

    "temperatura_aislamiento_bueno":
        temperatura * aislamiento_bueno,

    # --------------------------------------------------------
    # EQUIPAMIENTO
    # --------------------------------------------------------

    "equipos_por_persona": division_segura(
        cantidad_equipos,
        num_personas,
        valor_por_defecto=0
    ),

    "equipos_por_m2": division_segura(
        cantidad_equipos,
        superficie_m2,
        valor_por_defecto=0
    ),

    "carga_tecnologica":
        carga_tecnologica,

    "carga_tecnologica_por_persona": division_segura(
        carga_tecnologica,
        num_personas,
        valor_por_defecto=0
    ),

    "carga_tecnologica_por_m2": division_segura(
        carga_tecnologica,
        superficie_m2,
        valor_por_defecto=0
    ),

    "cantidad_equipos_alto_consumo":
        cantidad_equipos_alto_consumo,

    "equipos_alto_consumo_por_persona": division_segura(
        cantidad_equipos_alto_consumo,
        num_personas,
        valor_por_defecto=0
    ),

    "equipos_alto_consumo_por_m2": division_segura(
        cantidad_equipos_alto_consumo,
        superficie_m2,
        valor_por_defecto=0
    ),

    "carga_equipos_antiguos":
        cantidad_equipos * antiguedad_equipos,

    "carga_equipos_antiguos_por_persona": division_segura(
        cantidad_equipos * antiguedad_equipos,
        num_personas,
        valor_por_defecto=0
    ),

    "indice_obsolescencia_equipos": division_segura(
        antiguedad_equipos,
        antiguedad_inmueble,
        valor_por_defecto=0
    ),

    "brecha_antiguedad_inmueble_equipos":
        antiguedad_inmueble - antiguedad_equipos,

    # --------------------------------------------------------
    # ILUMINACIÓN
    # --------------------------------------------------------

    "proporcion_iluminacion_led":
        porcentaje_led,

    "cantidad_focos_led":
        cantidad_focos_led,

    "cantidad_focos_no_led":
        cantidad_focos_no_led,

    "focos_por_persona": division_segura(
        cantidad_focos,
        num_personas,
        valor_por_defecto=0
    ),

    "focos_por_m2": division_segura(
        cantidad_focos,
        superficie_m2,
        valor_por_defecto=0
    ),

    "carga_iluminacion_total":
        carga_iluminacion_total,

    "carga_iluminacion_led":
        carga_iluminacion_led,

    "carga_iluminacion_no_led":
        carga_iluminacion_no_led,

    "carga_iluminacion_por_persona": division_segura(
        carga_iluminacion_total,
        num_personas,
        valor_por_defecto=0
    ),

    "carga_iluminacion_por_m2": division_segura(
        carga_iluminacion_total,
        superficie_m2,
        valor_por_defecto=0
    ),

    "indice_ineficiencia_iluminacion":
        (1 - porcentaje_led) * horas_iluminacion,

    # --------------------------------------------------------
    # CONSUMO HISTÓRICO
    # --------------------------------------------------------

    "consumo_anterior_diario": division_segura(
        consumo_anterior,
        dias_facturacion,
        valor_por_defecto=0
    ),

    "consumo_anterior_por_persona": division_segura(
        consumo_anterior,
        num_personas,
        valor_por_defecto=0
    ),

    "consumo_anterior_por_m2": division_segura(
        consumo_anterior,
        superficie_m2,
        valor_por_defecto=0
    ),

    "consumo_anterior_por_equipo": division_segura(
        consumo_anterior,
        cantidad_equipos,
        valor_por_defecto=0
    ),

    "consumo_anterior_por_hora_actividad": division_segura(
        consumo_anterior,
        horas_uso_espacios_total,
        valor_por_defecto=0
    ),

    "consumo_anterior_ajustado_facturacion": (
        division_segura(
            consumo_anterior,
            dias_facturacion,
            valor_por_defecto=0
        )
        * 30
    ),

    "cambio_consumo_estimado":
        consumo_anterior * variacion_consumo,

    "consumo_estimado_tendencia":
        consumo_anterior * (1 + variacion_consumo),

    "variacion_consumo_absoluta_estimada":
        consumo_anterior * variacion_consumo,

    "variacion_consumo_magnitud":
        variacion_consumo.abs(),

    # --------------------------------------------------------
    # ENERGÍA SOLAR
    # --------------------------------------------------------

    "generacion_solar_diaria": division_segura(
        generacion_solar,
        dias_facturacion,
        valor_por_defecto=0
    ),

    "generacion_solar_por_persona": division_segura(
        generacion_solar,
        num_personas,
        valor_por_defecto=0
    ),

    "generacion_solar_por_m2": division_segura(
        generacion_solar,
        superficie_m2,
        valor_por_defecto=0
    ),

    "tasa_autogeneracion_solar_previa":
        tasa_autogeneracion_previa,

    "brecha_energetica_previa":
        brecha_energetica_previa,

    "saldo_solar_por_persona": division_segura(
        brecha_energetica_previa,
        num_personas,
        valor_por_defecto=0
    ),

    "saldo_solar_por_m2": division_segura(
        brecha_energetica_previa,
        superficie_m2,
        valor_por_defecto=0
    ),

    "dependencia_red_estimada": (
        1 - tasa_autogeneracion_previa
    ).clip(lower=0, upper=1),

    "tiene_generacion_solar":
        generacion_solar.gt(0).astype(np.int64),

    "intensidad_generacion_solar": division_segura(
        generacion_solar,
        superficie_m2,
        valor_por_defecto=0
    ),

    # --------------------------------------------------------
    # CORTES Y RESPALDO
    # --------------------------------------------------------

    "intensidad_cortes":
        dias_sin_electricidad * horas_respaldo,

    "frecuencia_cortes_relativa":
        frecuencia_cortes_relativa,

    "uso_respaldo_por_dia_corte": division_segura(
        horas_respaldo,
        dias_sin_electricidad,
        valor_por_defecto=0
    ),

    "uso_respaldo_diario": division_segura(
        horas_respaldo,
        dias_facturacion,
        valor_por_defecto=0
    ),

    "dependencia_respaldo":
        horas_respaldo.gt(0).astype(np.int64),

    "cobertura_respaldo_estimada":
        cobertura_respaldo_estimada,

    "cortes_con_planta":
        dias_sin_electricidad * respaldo_planta,

    "cortes_con_inversor":
        dias_sin_electricidad * respaldo_inversor,

    "cortes_con_panel_solar":
        dias_sin_electricidad * respaldo_solar,

    "cortes_sin_respaldo":
        dias_sin_electricidad * respaldo_ninguno,

    "indice_vulnerabilidad_electrica":
        indice_vulnerabilidad_electrica,

    # --------------------------------------------------------
    # TIPO DE INMUEBLE
    # --------------------------------------------------------

    "superficie_apartamento":
        superficie_m2 * apartamento,

    "superficie_casa":
        superficie_m2 * casa,

    "superficie_comercial":
        superficie_m2 * comercial,

    "ocupacion_apartamento":
        num_personas * apartamento,

    "ocupacion_casa":
        num_personas * casa,

    "ocupacion_comercial":
        num_personas * comercial,

    "equipos_apartamento":
        cantidad_equipos * apartamento,

    "equipos_casa":
        cantidad_equipos * casa,

    "equipos_comercial":
        cantidad_equipos * comercial,

    "actividad_comercial_estimada":
        horas_uso_espacios_total * comercial,

    "consumo_anterior_comercial":
        consumo_anterior * comercial,

    # --------------------------------------------------------
    # AISLAMIENTO Y ANTIGÜEDAD
    # --------------------------------------------------------

    "factor_aislamiento":
        factor_aislamiento,

    "antiguedad_aislamiento_malo":
        antiguedad_inmueble * aislamiento_malo,

    "antiguedad_aislamiento_regular":
        antiguedad_inmueble * aislamiento_regular,

    "antiguedad_aislamiento_bueno":
        antiguedad_inmueble * aislamiento_bueno,

    "demanda_termica_por_antiguedad":
        demanda_termica_estimada * antiguedad_inmueble,

    "indice_ineficiencia_constructiva":
        factor_aislamiento * antiguedad_inmueble,

    "inmueble_antiguo":
        antiguedad_inmueble.ge(30).astype(np.int64),

    "electrodomesticos_antiguos":
        antiguedad_equipos.ge(10).astype(np.int64),

    "inmueble_y_equipos_antiguos": (
        antiguedad_inmueble.ge(30)
        & antiguedad_equipos.ge(10)
    ).astype(np.int64),

    # --------------------------------------------------------
    # ESTACIONALIDAD
    # --------------------------------------------------------

    "mes_numero":
        mes_numero,

    "mes_sin":
        mes_sin,

    "mes_cos":
        mes_cos,

    "temperatura_estacional":
        temperatura * mes_sin,

    "aa_temporada_calida":
        carga_operativa_aa * grados_calor,

    "generacion_solar_estacional":
        generacion_solar * mes_sin,

    "consumo_anterior_estacional":
        consumo_anterior * mes_sin,

    # --------------------------------------------------------
    # HORARIO PICO
    # --------------------------------------------------------

    "pico_uso_diurno": (
        pico_manana + pico_tarde
    ).clip(upper=1).astype(np.int64),

    "pico_uso_nocturno": (
        pico_noche + pico_madrugada
    ).clip(upper=1).astype(np.int64),

    "aa_en_horario_calido":
        carga_operativa_aa * pico_tarde,

    "iluminacion_en_horario_nocturno": (
        carga_iluminacion_total
        * (
            pico_noche
            + pico_madrugada
        ).clip(upper=1)
    ),

    "actividad_en_horario_pico":
        horas_uso_espacios_total,

    # --------------------------------------------------------
    # ZONA
    # --------------------------------------------------------

    "temperatura_zona_costera":
        temperatura * zona_costera,

    "temperatura_zona_interior":
        temperatura * zona_interior,

    "temperatura_zona_suburbana":
        temperatura * zona_suburbana,

    "aa_zona_costera":
        carga_operativa_aa * zona_costera,

    "aa_zona_interior":
        carga_operativa_aa * zona_interior,

    "generacion_solar_zona_costera":
        generacion_solar * zona_costera,

    "generacion_solar_zona_interior":
        generacion_solar * zona_interior,

    "generacion_solar_zona_suburbana":
        generacion_solar * zona_suburbana,

    "vulnerabilidad_zona_costera":
        indice_vulnerabilidad_electrica * zona_costera,

    "vulnerabilidad_zona_interior":
        indice_vulnerabilidad_electrica * zona_interior,

    "vulnerabilidad_zona_suburbana":
        indice_vulnerabilidad_electrica * zona_suburbana,

    # --------------------------------------------------------
    # NIVEL SOCIOECONÓMICO
    # --------------------------------------------------------

    "equipos_nivel_alto":
        cantidad_equipos * nivel_alto,

    "equipos_nivel_medio":
        cantidad_equipos * nivel_medio,

    "equipos_nivel_bajo":
        cantidad_equipos * nivel_bajo,

    "aa_nivel_alto":
        carga_operativa_aa * nivel_alto,

    "aa_nivel_medio":
        carga_operativa_aa * nivel_medio,

    "aa_nivel_bajo":
        carga_operativa_aa * nivel_bajo,

    "superficie_nivel_alto":
        superficie_m2 * nivel_alto,

    "superficie_nivel_medio":
        superficie_m2 * nivel_medio,

    "superficie_nivel_bajo":
        superficie_m2 * nivel_bajo,

    "consumo_anterior_nivel_alto":
        consumo_anterior * nivel_alto,

    "consumo_anterior_nivel_medio":
        consumo_anterior * nivel_medio,

    "consumo_anterior_nivel_bajo":
        consumo_anterior * nivel_bajo,

    # --------------------------------------------------------
    # CERTIFICACIÓN
    # --------------------------------------------------------

    "certificacion_por_antiguedad":
        certificacion * antiguedad_inmueble,

    "certificacion_por_aislamiento":
        certificacion * factor_aislamiento,

    "certificacion_y_led":
        certificacion * porcentaje_led,

    "certificacion_y_consumo_anterior":
        certificacion * consumo_anterior
}


# ============================================================
# AGREGAR SOLO LAS BANDERAS DE DIVISIÓN NECESARIAS
# ============================================================

# Si no se detectó ningún denominador igual a cero,
# este diccionario estará vacío y no se agregará ninguna columna.
features.update(
    banderas_division_no_calculable
)


# ============================================================
# VARIABLES DERIVADAS DE LAS BANDERAS
# ============================================================

# Unificar las banderas de calidad existentes con las banderas
# de división que realmente fueron necesarias.
todas_las_banderas = {
    **banderas_calidad,
    **banderas_division_no_calculable
}

matriz_banderas = pd.DataFrame(
    todas_las_banderas,
    index=bd.index
)

cantidad_banderas_activas = (
    matriz_banderas.sum(axis=1)
)

cantidad_banderas_nulos = matriz_banderas[
    [
        "num_personas_era_nulo",
        "superficie_m2_era_nulo",
        "dias_facturacion_era_nulo",
        "cantidad_unidades_aa_era_nulo",
        "horas_uso_aa_dia_era_nulo",
        "cantidad_focos_era_nulo",
        "pct_iluminacion_led_era_nulo",
        "horas_uso_iluminacion_dia_era_nulo",
        "tiene_lavadora_era_nulo",
        "horas_dia_lavanderia_era_nulo",
        "generacion_solar_kwh_mensual_era_nulo",
        "consumo_kwh_mes_anterior_era_nulo"
    ]
].sum(axis=1)

cantidad_banderas_correccion = matriz_banderas[
    [
        "registro_con_correccion_deterministica",
        "personas_corregidas",
        "superficie_corregida",
        "cantidad_equipos_reconstruida"
    ]
].sum(axis=1)

cantidad_banderas_inconsistencia = matriz_banderas[
    [
        "equipos_total_inconsistente",
        "aa_inconsistente",
        "iluminacion_inconsistente",
        "respaldo_inconsistente"
    ]
].sum(axis=1)

cantidad_banderas_consumo = matriz_banderas[
    [
        "inicio_consumo_desde_cero",
        "sin_consumo_dos_meses",
        "caida_consumo_a_cero"
    ]
].sum(axis=1)

features.update({

    "cantidad_banderas_activas":
        cantidad_banderas_activas.astype(np.int64),

    "cantidad_variables_imputadas":
        cantidad_banderas_nulos.astype(np.int64),

    "cantidad_correcciones_deterministicas":
        cantidad_banderas_correccion.astype(np.int64),

    "cantidad_inconsistencias":
        cantidad_banderas_inconsistencia.astype(np.int64),

    "cantidad_anomalias_consumo":
        cantidad_banderas_consumo.astype(np.int64),

    "tiene_alguna_inconsistencia":
        cantidad_banderas_inconsistencia.gt(0).astype(np.int64),

    "tiene_alguna_variable_imputada":
        cantidad_banderas_nulos.gt(0).astype(np.int64),

    "tiene_alguna_correccion":
        cantidad_banderas_correccion.gt(0).astype(np.int64),

    "inconsistencia_operativa": (
        matriz_banderas["aa_inconsistente"]
        | matriz_banderas["iluminacion_inconsistente"]
        | matriz_banderas["respaldo_inconsistente"]
    ).astype(np.int64),

    "inconsistencia_consumo":
        cantidad_banderas_consumo.gt(0).astype(np.int64),

    "patron_consumo_cero": (
        matriz_banderas["inicio_consumo_desde_cero"]
        | matriz_banderas["sin_consumo_dos_meses"]
        | matriz_banderas["caida_consumo_a_cero"]
    ).astype(np.int64),

    "patron_cambio_extremo_consumo":
        variacion_consumo.abs().ge(1).astype(np.int64),

    "consumo_historico_anomalo": (
        cantidad_banderas_consumo.gt(0)
        | variacion_consumo.abs().ge(1)
    ).astype(np.int64),

    "registro_consumo_no_confiable": (
        cantidad_banderas_consumo.gt(0)
        | matriz_banderas[
            "consumo_kwh_mes_anterior_era_nulo"
        ].eq(1)
        | variacion_consumo.abs().ge(1)
    ).astype(np.int64),

    "indice_calidad_registro": (
        1
        - division_segura(
            cantidad_banderas_activas,
            len(todas_las_banderas),
            valor_por_defecto=0
        )
    ).clip(lower=0, upper=1)
})


# ============================================================
# DATAFRAME DE VARIABLES NUEVAS
# ============================================================

df_features = pd.DataFrame(
    features,
    index=bd.index
)

df_features = df_features.replace(
    [np.inf, -np.inf],
    np.nan
)


# ============================================================
# VALIDAR NULOS ANTES DE AGREGAR
# ============================================================

nulos_features = (
    df_features
    .isna()
    .sum()
    .loc[lambda serie: serie.gt(0)]
    .sort_values(ascending=False)
)

if not nulos_features.empty:

    print("\nVariables derivadas que todavía contienen nulos:")

    display(
        nulos_features.to_frame(
            "cantidad_nulos"
        )
    )

    print(
        "Total de nulos en variables derivadas:",
        int(nulos_features.sum())
    )

else:

    print(
        "\nNo se encontraron valores nulos "
        "en las variables derivadas."
    )


# ============================================================
# EVITAR COLUMNAS DUPLICADAS
# ============================================================

columnas_reemplazadas = [
    columna
    for columna in df_features.columns
    if columna in bd.columns
]

bd = bd.drop(
    columns=columnas_reemplazadas,
    errors="ignore"
)


# ============================================================
# AGREGAR TODO EN UNA SOLA OPERACIÓN
# ============================================================

bd = pd.concat(
    [bd, df_features],
    axis=1
).copy()


# ============================================================
# NORMALIZAR LAS BANDERAS DE DIVISIÓN CREADAS
# ============================================================

nombres_banderas_division = list(
    banderas_division_no_calculable.keys()
)

for bandera in nombres_banderas_division:

    bd[bandera] = (
        pd.to_numeric(
            bd[bandera],
            errors="coerce"
        )
        .fillna(0)
        .astype(np.int64)
    )

    assert bd[bandera].isna().sum() == 0, (
        f"La bandera '{bandera}' contiene valores nulos."
    )

    assert bd[bandera].isin([0, 1]).all(), (
        f"La bandera '{bandera}' contiene "
        "valores diferentes de 0 y 1."
    )


# ============================================================
# MENSAJE FINAL
# ============================================================

variables_nuevas = [
    columna
    for columna in df_features.columns
    if columna not in columnas_iniciales
]

print("\n" + "=" * 70)
print("FEATURE ENGINEERING COMPLETADO")
print("=" * 70)
print(f"Filas procesadas: {bd.shape[0]:,}")
print(f"Variables iniciales: {len(columnas_iniciales)}")
print(f"Variables calculadas: {df_features.shape[1]}")
print(f"Variables nuevas agregadas: {len(variables_nuevas)}")
print(f"Variables existentes reemplazadas: {len(columnas_reemplazadas)}")
print(f"Banderas de división creadas: {len(nombres_banderas_division)}")
print(f"Total de variables finales: {bd.shape[1]}")
print("=" * 70)

if nombres_banderas_division:

    print("\nBanderas de divisiones no calculables creadas:")

    for bandera in nombres_banderas_division:

        cantidad_casos = int(
            bd[bandera].sum()
        )

        porcentaje_casos = (
            bd[bandera].mean() * 100
        )

        print(
            f"- {bandera}: "
            f"{cantidad_casos:,} casos "
            f"({porcentaje_casos:.4f} %)"
        )

else:

    print(
        "\nNo se detectaron divisiones entre cero. "
        "No se creó ninguna bandera de división."
    )


# ============================================================
# VALIDACIONES FINALES
# ============================================================

columnas_numericas = (
    bd.select_dtypes(include=np.number)
)

infinitos = int(
    np.isinf(columnas_numericas)
    .sum()
    .sum()
)

nulos_finales = (
    bd.isna()
    .sum()
    .loc[lambda serie: serie.gt(0)]
    .sort_values(ascending=False)
)

variables_constantes = [
    columna
    for columna in df_features.columns
    if df_features[columna].nunique(
        dropna=False
    ) <= 1
]

print(
    f"\nValores infinitos encontrados: "
    f"{infinitos}"
)

print(
    f"Variables nuevas constantes: "
    f"{len(variables_constantes)}"
)

print(
    f"Total de valores nulos finales: "
    f"{int(nulos_finales.sum())}"
)

if variables_constantes:

    print("\nLista de variables constantes:")

    for variable in variables_constantes:
        print(f"- {variable}")

if not nulos_finales.empty:

    print("\nVariables con valores nulos finales:")

    display(
        nulos_finales.to_frame(
            "cantidad_nulos"
        )
    )

else:

    print(
        "\nLa base final no contiene valores nulos."
    )

Se crearon correctamente 55 alias y variables auxiliares.

No se encontraron valores nulos en las variables derivadas.

FEATURE ENGINEERING COMPLETADO
Filas procesadas: 94,537
Variables iniciales: 77
Variables calculadas: 161
Variables nuevas agregadas: 161
Variables existentes reemplazadas: 0
Banderas de división creadas: 2
Total de variables finales: 238

Banderas de divisiones no calculables creadas:
- division_por_antiguedad_inmueble_no_calculable: 2,148 casos (2.2721 %)
- division_por_dias_sin_electricidad_no_calculable: 2,769 casos (2.9290 %)

Valores infinitos encontrados: 0
Variables nuevas constantes: 5
Total de valores nulos finales: 0

Lista de variables constantes:
- cantidad_variables_imputadas
- cantidad_anomalias_consumo
- tiene_alguna_variable_imputada
- inconsistencia_consumo
- patron_consumo_cero

La base final no contiene valores nulos.


## EXPORTACIÓN

In [ ]:
# Guardar el DataFrame bd en el mismo directorio del notebook

nombre_archivo = "03_feature_engineering.csv"

bd.to_csv(
    nombre_archivo,
    index=False,
    encoding="utf-8-sig"
)

print(f"Base de datos guardada correctamente como: {nombre_archivo}")

Base de datos guardada correctamente como: bd_feature_engineering.csv
